# Testando o modelo do Detector de Emoções

## Etapa 1 - Importando as bibliotecas

In [ ]:
import cv2
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow
import zipfile

In [ ]:
%tensorflow_version 2.x

In [ ]:
import tensorflow
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
tensorflow.__version__

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
# COnectar com o drive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# arquivo com a localização do material de aula zipado
path = '/content/gdrive/My Drive/Material.zip'
zip_object = zipfile.ZipFile(file = path, mode = 'r')
# Extrai o material de aula pra a raiz
zip_object.extractall('./')
zip_object.close

In [ ]:
# Mostra a image
imagem = cv2.imread('Material/testes/teste_gabriel.png')
cv2_imshow(imagem)

In [ ]:
# Formato da imagem
imagem.shape

## Testando o Detector

### Carregamento dos modelos

In [ ]:
cascade_faces = "/content/Material/haarcascade_frontalface_default.xml"
caminho_modelo = "/content/Material/modelo_01_expressoes.h5"
face_detection = cv2.CascadeClassifier(cascade_faces)
classificador_emocoes = load_model(caminho_modelo, compile = False)
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

### Detecção de faces

In [ ]:
# Cria uma cópia da imagem
original = imagem.copy()
faces = face_detection.detectMultiScale(original, scaleFactor = 1.1,
                                        minNeighbors = 3, minSize = (20,20))

In [ ]:
faces # para mostrar a posição da face
# x e y é onde esta o inicio do quadrado que foi detectado a face
# os outros dois números são o tamanho do quadrado/retangulo que a esta a face

In [ ]:
len(faces)

In [ ]:
faces.shape

### Extração do ROI (region of interest)

In [ ]:
# altera a cor da imagem para escala de cinza
cinza = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
cv2_imshow(cinza)

In [ ]:
cinza.shape

In [ ]:
# Uma imagem com a regiao delimitada de onde esta a face
roi = cinza[40:40 + 128, 162:162 + 128] # os valores foram encontrados lá em detecção de face

In [ ]:
cv2_imshow(roi)

In [ ]:
# formato da imagem de interesse
roi.shape

In [ ]:
roi # vai mostrar uma matriz com os valores dos pixels

In [ ]:
# Rediminsiona pra 48x48
roi = cv2.resize(roi, (48, 48))
cv2_imshow(roi) # mostra a imagem

In [ ]:
# tipo de dado
roi.dtype

In [ ]:
# converte para tipo float
roi = roi.astype('float')
roi.dtype

In [ ]:
roi

In [ ]:
# faz uma normalização da img
roi = roi / 255

In [ ]:
roi

In [ ]:
# converte para uma array
roi = img_to_array(roi)

In [ ]:
roi

In [ ]:
roi.shape

In [ ]:
# Expande as dimensões
roi = np.expand_dims(roi, axis = 0)

In [ ]:
roi.shape # o resultado do shape é basicamente, 1 img de 48x48 com 1 canal de cor

### Previsões

In [ ]:
# para retornar a probabilidade de cada uma das emocões na lista
preds = classificador_emocoes.predict(roi)[0]

In [ ]:
# resultado
preds

In [ ]:
len(preds)

In [ ]:
# emocao com mais probabilidade
emotion_probability = np.max(preds)
emotion_probability

In [ ]:
# indice da lista com a maior probabilidade
preds.argmax()

In [ ]:
# mostra o label correspondente
label = expressoes[preds.argmax()]
label

### Resultados

In [ ]:
# Escreve o label na imagem
cv2.putText(original, label, (162, 40 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65,
            (0, 0, 255), 2, cv2.LINE_AA)
# Desenha um retangulo em volta da face
cv2.rectangle(original, (162, 40), (162 + 128, 40 + 128), (0, 0, 255), 2)
cv2_imshow(original) # Mostra a imagem

In [ ]:
# Para mostrar na barra de porcentagem de quanto que vai mostrar de cada emocao identificada
probabilidades = np.ones((250,300,3), dtype= 'uint8') * 255
probabilidades

In [ ]:
probabilidades.shape

In [ ]:
# Mostra a imagem
cv2_imshow(original)
if len(faces) == 1: # Se tiver só uma face
  # Tecnicamente vai mostrar a probabilidade de cada emocao para cada emocao
  for (i, (emotion, prob)) in enumerate(zip(expressoes, preds)):
    #print(i, emotion, prob)
    # Formata o float para algo como 0,00%
    text = "{}: {:.2f}%".format(emotion, prob * 100)
    w = int(prob * 300)
    # desenha um retangulo com a probabilidade
    cv2.rectangle(probabilidades, (7, (i * 35) + 5), (w, (i * 35) + 35), (200, 250, 20), -1)
    # Escreve o texto correspodente de cada emocao
    cv2.putText(probabilidades, text, (10, (i * 35) + 23), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)
# Printa as probalidades
cv2_imshow(probabilidades)